# NQS tutorial: the 2D Ising transition

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PhilipVinc/Lectures/blob/main/2606_NQS-Ising-tutorial/ising_nqs_exercise.ipynb)

The goal today is to use Neural Quantum States (NQS) to study the ground state of the 2D
transverse-field Ising model (TFIM) on a square lattice,

$$ H = -h\sum_i \sigma^x_i + J\sum_{\langle i j\rangle}\sigma^z_i\sigma^z_j,\qquad J=1,$$

which has a quantum phase transition at $h_c/J \approx 3.044$ between a
ferromagnet ($h<h_c$) and a paramagnet ($h>h_c$). 
The notebook is divided into 2 parts: in part 1 you will be more exploring how different
hyperparameters and architectures affect the accuracy of a sample calculation, while in 
part 2 you will be fixing on an optimization protocol and using it to study the physics of the model.

This notebook is thought to run on your laptop's CPU, but if you have some GPUs somewhere, 
it would be great to use them because it will sensibly speedup the calculations.

Two important notes: 
- You are adults. I expect you know how to search online on netket's documentation for what you can do with it. Not everything is explained here.
- this notbook is a sketch: feel free to pick a different model (Heisenberg for example) or to apply what you learned to different problems...

---
*Practical notes.*
- On **Google Colab**, just run the setup cell below: it installs everything and downloads `ansatze.py` for you.
- Running **locally**: you need `netket>=3.22.2`, `nqxpack` and `matplotlib` (see the `pyproject.toml` / `README.md` next to this notebook). Check your version with `netket.__version__` and update if necessary.
- Run this notebook **from its own directory** so that `import ansatze` works (the custom models live in `ansatze.py` — required by `nqxpack`).
- A note about the architectures: The Convolution runtime of jax is extremely slow on CPUs, so be prepared to 'ignore' or 'skip' calculations with the CNN if they start to feel too slow...
 

In [ ]:
# === Run this cell first on Google Colab ===
# It installs the dependencies and downloads the custom `ansatze.py` module.
# Running locally instead? Skip it: install from pyproject.toml and run this
# notebook from its own directory.
try:
    import google.colab  # noqa: F401  -> we are on Colab
    %pip install --quiet "netket==3.22.2" nqxpack matplotlib
    !wget -q -nc https://raw.githubusercontent.com/PhilipVinc/Lectures/main/2606_NQS-Ising-tutorial/ansatze.py
except ImportError:
    pass

## Part 0 — a guided tour of NetKet 

To start, I want to walk you through one full VMC calculation, line by line: 
 - build a model,
 - optimise it
 - rate the optimization accuracy and ovtain accurate estimates of observables 
 - impose a symmetry.
 
At the end i will 'compress' the boilerplate into two small helper functions (`make_vstate`, `run_vmc`) 
that you will reuse throughout Parts 1 and 2.

In [ ]:
import os
os.environ["NETKET_SHARDING"] = "0"   # benchmark on a single (multi-core) CPU

import json
import numpy as np
import matplotlib.pyplot as plt

import jax
import netket as nk
import nqxpack

from ansatze import CNN, ViT   # custom architectures (importable -> nqxpack-saveable)

os.makedirs("logs", exist_ok=True)     # JsonLog output
os.makedirs("states", exist_ok=True)   # nqxpack checkpoints

print("netket", nk.__version__, "| devices:", jax.devices())

### The lattice, the Hilbert space and the Hamiltonian

A NetKet calculation always starts from three objects: a **graph** (the lattice),
a **Hilbert space** (defining the computational basis we use and what samples can be generated), 
and an **hamiltonian** which defines the energy we will minimise. For $4\times4$ the problem is 
small enough that exact diagonalisation gives the reference ground-state energy.

Note that netket does not use 'standard' matrices, instead they use a _sort-of-sparse-format_ designed
to work efficiently with VMC calculations. Nevertheless, for those small systems you can always convert
them to matrices by calling `operator.to_sparse()`.

FYI, you can visualize the lattice with `lattice.draw()`.

In [ ]:
L = 4
g = nk.graph.Square(L, pbc=True)              # 2D square lattice, periodic BC
hi = nk.hilbert.Spin(s=1 / 2, N=g.n_nodes)    # one spin-1/2 per site; values in {-1,+1}
H = nk.operator.Ising(hilbert=hi, graph=g, h=3.0, J=1.0)   # h ~ h_c: the hard point

E_gs = nk.exact.lanczos_ed(H, k=1)[0]         # exact reference (only feasible for small N)
print(f"N = {g.n_nodes} spins,  exact ground-state energy  E_gs = {E_gs:.6f}")

### A full VMC calculation

A variational Monte Carlo (VMC) ground-state search needs four ingredients: a
**model** (the wave-function ansatz $\psi_\theta$), a **sampler** (to draw spin
configurations $\propto|\psi_\theta|^2$), a **variational state** (model +
sampler + parameters), and an **optimization driver** which determines the
optimization algorithm. 

The central element in NetKet is the `nk.vqs.MCState`, which we call the **variational state**.
The variational state behaves almost as a ket: you can :
- you can call `vstate.samples` to generate `vstate.n_samples` samples from the born amplitude
- call `vstate.expect(operator)` to compute expectation values using those samples, or 
`vstate.expect_to_precision(operator, atol=1e-4)` to get an expectation value at the defined accuracy,
which means that it will continue sampling until the accuracy is reached.
- call `vstate.expect_and_grad(operator)` to get expectation value and gradient of this operator
- call `vstate.quantum_geometric_tensor()` to get the S matrix/quantum geomtric tensor.
- call `vstate.log_value(samples)` to get amplitudes, or `vstate.variables` to get its parameters.

Later on we will show that it's also possible to transform those variational states for example by projecting them to
a given symmetry subsector.

In [ ]:
# 1. MODEL: a small Restricted Boltzmann Machine. `alpha` = hidden units per spin, total number
# of parameters is roughly alpha N^2.
model = nk.models.RBM(alpha=1, param_dtype=float)

# 2. SAMPLER: Metropolis with single spin flips. The TFIM does NOT conserve the
#    magnetisation, so single flips are the right move. Many parallel chains keep
#    the chains short (n_samples / n_chains each).
sampler = nk.sampler.MetropolisLocal(hi, n_chains=128)

# 3. VARIATIONAL STATE: ties the model to the sampler. It draws `n_samples`
#    configurations per optimisation step; `n_discard_per_chain` drops a few
#    warm-up samples every time you change the parameters. Put 0 to be more efficient,
#    but sometimes it can help to have a small number here like 1 or 2. It's not really
#    Kosher according to MCMC dogmas, but it works.
vstate = nk.vqs.MCState(sampler, model, n_samples=1024, n_discard_per_chain=1)

# 4. DRIVER: minimise <H> by Stochastic Reconfiguration (natural-gradient VMC).
#    `optimizer` is the optimization algorithm used to update the parameters. While
#        you could use non-sgd ones, that's the only one that makes mathematical sense
#        with SR. 
#    `diag_shift` regularises the (quantum) geometric tensor as S -> S+shift * I
#    `linear_solver` defines the algorithm to use to solve the lienar system
#        (S+shift * I)θ=∇E . The most accurate and reliable one is `pinv_smooth`,
#        while conjugate gradients is the worst. cholesky with a fallback is a 
#        good middle ground. 
#    `momentum` is a terrible name for an exponential smoothing of the geometric tensor
#        across different iterations. It's quite useful at times to stabilize calculations
#        but we still are not very good at tuning it. Usually we start with it disabled.
optimizer = nk.optimizer.Sgd(learning_rate=0.02)
gs = nk.driver.VMC_SR(H, 
                      optimizer, 
                      diag_shift=1e-3, 
                      linear_solver=nk.optimizer.solver.cholesky_with_fallback,
                      momentum=None,
                      variational_state=vstate)

# RUN: optimise for 200 steps, recording the energy trace in memory. Alternative is
# JsonLog which saves to a file.
log = nk.logging.RuntimeLog()
gs.run(n_iter=200, out=log)
print("done — final logged energy:", log.data["Energy"].Mean[-1].real)

### Reading and plotting the energy trace

`log.data["Energy"]` is a `History` object with `.iters` and `.Mean`. The energy
should fall towards `E_gs` and then fluctuate around it (Monte-Carlo noise).

To read json files you can do `nk.utils.history.HistoryDict.from_file("filename.log")` and it will
work the same way.

In [ ]:
energy = log.data["Energy"]
plt.figure(figsize=(6, 4))
plt.plot(np.asarray(energy.iters), np.asarray(energy.Mean).real, label="VMC energy")
plt.axhline(E_gs, ls="--", c="k", label="exact $E_{gs}$")
plt.xlabel("VMC step"); plt.ylabel("energy"); plt.legend(); plt.tight_layout(); plt.show()

### An accurate final energy — `expect_to_precision`

The last value of the trace is a single noisy estimate which might be inaccurate. To get an accurate number 
we resample the converged state until the **relative** error of the mean drops below
`rtol`. It returns running statistics; use `.mean` and `.get_stats().error_of_mean`.

In [ ]:
stats = vstate.expect_to_precision(H, rtol=1e-3, verbose=False)
E, err = stats.get_stats().mean.real, stats.get_stats().error_of_mean
print(f"E = {E:.5f} ± {err:.1e}   (relative error vs exact: {abs(E - E_gs)/abs(E_gs):.2e})")

### Scoring the quality — the V-score

The **V-score** $V = N\,\mathrm{Var}(H)/(\langle H\rangle-E_\infty)^2$ is a
size-intensive, ansatz- and basis-independent quality metric: it vanishes for an
exact eigenstate and lets us compare calculations when we don't have the exact
benchmark.

`nk.observable.VScore` computes exactly this in one pass — prefactor $N$ included
(it reads the size off the Hilbert space), so the number it returns *is* the
V-score, no rescaling. The only thing you supply is $E_\infty$ via
`trace_diagonal`; for the TFIM the diagonal is traceless, so
$E_\infty=\texttt{trace\_diagonal}=0$.

In [ ]:
V = vstate.expect(nk.observable.VScore(H, trace_diagonal=0.0)).mean.real
print(f"V-score = {V:.3e}")

### Imposing a symmetry — the $k=0$ projection

The ground state has zero momentum. We can *enforce* this by applying the Bloch
projector $P_0=\tfrac1{|G|}\sum_{g\in G}T_g$ onto the $k=0$ sector, built from the
lattice translation group. `nk.vqs.apply_operator` wraps any variational state
into the symmetrised one (at a cost $\times|G|$ per amplitude).

In [ ]:
rep = nk.symmetry.canonical_representation(hi, g.translation_group())
P0 = rep.projector(k=(0.0, 0.0))                 # zero-momentum sector
print(f"|G| = {len(P0.operators)} translations in the projector")

vstate_sym = nk.vqs.apply_operator(P0, vstate)   # same parameters, now symmetrised
print(f"E (plain) = {vstate.expect(H).mean.real:.5f}")
print(f"E (k=0)   = {vstate_sym.expect(H).mean.real:.5f}")

### Distilling the boilerplate into helpers

That is the whole workflow. We now wrap the repetitive parts into two helpers and
switch the logger to **`nk.logging.JsonLog`**, which streams the energy (and the
parameters) to disk as `logs/<name>.log` (+ `.mpack`) so runs can be compared and
reloaded later.

Feel free to change those helpers if you want.

In [ ]:
def make_vstate(model, hilbert=hi, n_samples=1024, n_chains=128, seed=0):
    sampler = nk.sampler.MetropolisLocal(hilbert, n_chains=n_chains)
    return nk.vqs.MCState(sampler, model, n_samples=n_samples,
                          n_discard_per_chain=1, seed=seed)

def run_vmc(vstate, *, name, H=H, n_iter=300, lr=0.02, diag_shift=1e-3):
    """Optimise `vstate` and stream the energy to logs/<name>.log (+ .mpack)."""
    optimizer = nk.optimizer.Sgd(learning_rate=lr)
    driver = nk.driver.VMC_SR(H, optimizer, diag_shift=diag_shift,
                              variational_state=vstate)
    driver.run(n_iter=n_iter, out=nk.logging.JsonLog(f"logs/{name}", "w"))
    return vstate

Read a `JsonLog` back and overlay several convergence curves. The `_col`
helper copes with complex-valued models (the ViT) storing the energy as a
`{"real":…, "imag":…}` dict.

In [ ]:
def _col(x):
    return np.array(x["real"]) if isinstance(x, dict) else np.array(x)

def load_energy(name):
    with open(f"logs/{name}.log") as f:
        data = json.load(f)
    return np.array(data["Energy"]["iters"]), _col(data["Energy"]["Mean"])

def plot_convergence(names, E_ref=E_gs):
    plt.figure(figsize=(6, 4))
    for nm in names:
        it, e = load_energy(nm)
        plt.plot(it, np.abs(e - E_ref) / abs(E_ref), label=nm)
    plt.yscale("log")
    plt.xlabel("VMC step"); plt.ylabel(r"$|E - E_{gs}| / |E_{gs}|$")
    plt.legend(); plt.title("convergence"); plt.tight_layout(); plt.show()

## Part 1 — architectures and hyperparameters *(4×4, $h=3$)*

For each ansatz family: build the state, `run_vmc`, save it with `nqxpack.save`,
and record a `final_score`. At the end you will overlay all convergence curves and
compare the final Energy / V-scores.

To compute the final score, write a function that uses `expect_to_accuracy(operator, [rtol/atol=?]) ` to compute those quantities with the desired accuracy, as the estimate with the number of samples used for training is often not extraordinarily precise.

The recipe is always the same:
```python
vs = make_vstate(<model>)
run_vmc(vs, name="<tag>", n_iter=300)
nqxpack.save(vs, "states/<tag>.nk")
final_score(vs, "<label>")
```

In [ ]:
# TODO: write final_score(vstate, name) and return a dict with the numbers.
#   - accurate energy + error: vstate.expect_to_precision(H, rtol=...)  (NOT plain expect)
#   - V-score: vstate.expect(nk.observable.VScore(H, trace_diagonal=0.0)).mean.real  (see Part 0)
#   - print a one-line summary (energy ± err, rel. err vs E_gs, V-score)
def final_score(vstate, name="", *, H=H, rtol=1e-3, E_ref=E_gs):
    ...

### 1.1 — Restricted Boltzmann Machine

`nk.models.RBM(alpha=…, param_dtype=float)`. The density `alpha` sets the number
of hidden units ($=\alpha N$). Worked example for `alpha=1` is given.

In [ ]:
# TODO: do the same for alpha = 2 and alpha = 4 (loop over the two values).
# Use names "rbm_a2", "rbm_a4" so the convergence plot can find them.
for alpha in (2, 4):
    ...

### 1.2 — Multi-layer perceptron

`nk.models.MLP(hidden_dims=(…,), param_dtype=float)`. The tuple length is the
number of hidden layers (use 2–3, e.g. `(32, 32)`); the output is already a
scalar log-amplitude.

In [ ]:
# TODO: train an MLP with a few hidden layers (e.g. hidden_dims=(32, 32)).
#   build -> run_vmc(name="mlp") -> nqxpack.save -> final_score
...

### 1.3 — Convolutional network (residual)

`from ansatze import CNN`. 

This architecture stacks residual blocks of circular-padding convolutions (periodic BC) 
and **sums over space**, so it is *translation invariant by construction*. 

Parameters: 
- `features`: the number of features/channels in each layer. Similar to alpha in the RBM or d_model in the Vision Transformer.
- `depth` : number of layers. (keep `depth` small — 2 residual blocks already work well and deeper nets are harder to
optimise with SR on this small lattice). 

*Note: convolutions are comparatively slow inside the Metropolis sampler on CPU, so this is going to be slow.*

In [ ]:
# TODO: train a CNN, e.g. CNN(features=8, depth=2). name="cnn".
...

### 1.4 — Vision Transformer (factored attention, 2×2 patches)

The Vision transformer can be imported as `from ansatze import ViT`. 

This architecture relies on patching, where the $4\times4$ lattice is cut into 2×2 patches (therefore 4 total patches)
and processed by factored multi-head attention (see the `ViT-wave-function` tutorial in NetKet website). 
Parameters: 
- `num_layers`: nothing particular to say
- `d_model`: the size of the vector space on which it operates. That's the main control knob for accuracy;
- `n_heads`: controls the number of parameters. This must be between 1 and `d_model`. If it's 1, we get 1 set of parameters for the whole vector space, while if it's `d_model` we get 1 different set of parameters for every dimension. In general we define it as a ratio of approximately 1/6 of the `d_model`.

Note that wrt other architectures, you will need to tune the diag shift and learning rate.

This architecture is partly invariant, under translations of the patches (so all translations of multiple 2 along each direction). You can 'complete the simmetry', but it's a bit tricky. Look at [this tutorial](https://netket.readthedocs.io/en/latest/tutorials/symmetries/iterative_symmetrization.html#application-patch-based-neural-networks) if you are interested.

In [ ]:
# TODO: train a ViT, e.g. ViT(num_layers=2, d_model=24, n_heads=4). name="vit".
...

### 1.5 — Compare

Overlay every convergence curve (log scale) and look at the final scores printed
above.

In [ ]:
# TODO: call plot_convergence([...]) with all the run names you trained.
...

**Discuss.**
- Rank the ansätze by the lowest energy / V-score they reach. Does more
  expressive power always help at fixed compute budget?
- The RBM/MLP have no built-in geometry; the CNN bakes in locality + translation
  symmetry; the ViT mixes patches through attention. How does that show up in the
  curves?
- Cost vs accuracy: count parameters
  (`nk.jax.tree_size(vs.parameters)`) and relate to wall-clock.

## Part 1b — sharpening with a $k=0$ projection

We met the $k=0$ Bloch projector in Part 0. Here we put it to work: take a
**non-symmetric** ansatz (the RBM is ideal — unlike the CNN it is not already
translation invariant), project it, retrain, and compare the final score to the
unprojected run.

```python
vs_sym = nk.vqs.apply_operator(P0, vs)   # P0 from Part 0; cost x |G|
run_vmc(vs_sym, name="...")              # VMC_SR optimises the symmetrised state
```


In [ ]:
# TODO:
#  1. build a fresh RBM state (alpha=2) and project it with nk.vqs.apply_operator(P0, vs)
#  2. run_vmc(vs_sym, name="rbm_a2_k0"); save; final_score
#  3. compare against an un-projected RBM alpha=2 (final score + convergence plot)
...

**Discuss.** How much does the projection gain in energy / V-score? The
cost grew by $|G|=16$ — was it worth it? *(For large systems the $\times|G|$ cost
is prohibitive; the cheaper route is to bake the symmetry into the architecture —
e.g. the `transl_invariant=True` ViT — and use coset/iterative refinement, see
the `symmetries/iterative_symmetrization` tutorial. The CNN is already $k=0$ by
construction, so projecting it would be redundant.)*

**— end of Part 1 —**

## Part 2 — across the transition at $8\times8$

Pick **one** architecture and study the physics as a function of $h$.

If you pick a non-symmetric ansatz, wrap it with
`nk.vqs.apply_operator(P0_2, vs)` exactly as in Part 1b at the end (or even during training)
to get more accurate solutions.

In [ ]:
# given: the 8x8 system and the field grid straddling h_c ~ 3.044
L2 = 8
g2 = nk.graph.Square(L2, pbc=True)
hi2 = nk.hilbert.Spin(s=1 / 2, N=g2.n_nodes)
h_values = [2.0, 2.75, 3.044, 3.5, 4.25]

def H_at(h):
    return nk.operator.Ising(hilbert=hi2, graph=g2, h=h, J=1.0)

print(f"N = {g2.n_nodes} spins  (too large for ED)")

Train one state per field value and keep them in a dict. (Reduce `n_iter`
if you are CPU-bound; convergence need not be perfect to see the physics.)

In [ ]:
# TODO: for each h in h_values, train a CNN on the 8x8 lattice and store it.
#   model = CNN(features=8, depth=2)
#   vs = make_vstate(model, hilbert=hi2, n_samples=2048)
#   run_vmc(vs, name=f"L8_h{h}", H=H_at(h), n_iter=400)
#   nqxpack.save(vs, f"states/L8_h{h}.nk");  states[h] = vs
states = {}
...

### 2.1 — Spin–spin correlations

Measure the connected correlator
$C(r) = \langle\sigma^z_0\,\sigma^z_r\rangle - \langle\sigma^z_0\rangle\langle\sigma^z_r\rangle$
along a lattice axis with `nk.observable.ConnectedCorrelator`, built from
single-site $\sigma^z$ operators `nk.operator.spin.sigmaz(hilbert, i)`. The helper
maps a displacement $(dx,dy)$ to a site index.

In [ ]:
# given: geometry + correlator builder
_pos = np.asarray(g2.positions).astype(int)
_lookup = {(int(x) % L2, int(y) % L2): i for i, (x, y) in enumerate(_pos)}

def site(dx, dy):
    return _lookup[(dx % L2, dy % L2)]

def correlators_along_x():
    """ConnectedCorrelator(sigma^z_(0,0), sigma^z_(r,0)) for r = 1 .. L2//2."""
    sz0 = nk.operator.spin.sigmaz(hi2, site(0, 0))
    rs = list(range(1, L2 // 2 + 1))
    ops = [nk.observable.ConnectedCorrelator(sz0, nk.operator.spin.sigmaz(hi2, site(r, 0)))
           for r in rs]
    return rs, ops

In [ ]:
# TODO: for each trained state, measure C(r) along x with expect_to_precision
#   (rtol ~ 0.05 is enough), then plot |C(r)| vs r (semilog-y) for every h.
# Hint: vs.expect_to_precision(ops, rtol=0.05) accepts a LIST of observables and
#       returns a list of statistics; use .mean.real for each.
rs, ops = correlators_along_x()
...

### 2.2 — Rényi-2 entanglement entropy

`nk.observable.Renyi2EntanglementEntropy(hilbert, partition=[…site indices…])`
estimates $S_2 = -\log\mathrm{Tr}\rho_A^2$ for subsystem $A$. Use a half-system
cut (left half of the lattice).

In [ ]:
# given: half-system partition (left half: columns x < L2/2)
partition_A = [i for i, (x, y) in enumerate(_pos) if x < L2 // 2]
print(f"|A| = {len(partition_A)} of {g2.n_nodes} sites")
S2_obs = nk.observable.Renyi2EntanglementEntropy(hi2, partition=partition_A)

In [ ]:
# TODO: measure S2 for each trained state and plot S2 vs h.
#   use vs.expect(S2_obs) ; take .mean.real and .error_of_mean
...